#### Oliwia Obydź
### **Wycena nieruchomości na rynku wtórnym w województwie mazowieckim**

#### **1. Temat projektu**

Tematem projektu jest wycena nieruchomości na rynku wtórnym w województwie mazowieckim. Projekt oparty jest na danych z portalu otodom.pl. Głównym celem jest zbadanie wpływu parametrów technicznych oraz lokalizacji na wartość nieruchomości. Na ich podstawie zaimplementowany zostanie model uczenia maszynowego zdolny do estymowania cen, co pozwoli na weryfikację ofert rynkowych pod względem zawyżenia lub zaniżenia ceny.

#### **2. Skład zespołu**

Projekt indywidualny: Oliwia Obydź, numer albumu: 227758

#### **3. Opis zbioru danych**

**Źródło danych:** Dane zostały pozyskane metodą web scrapingu z polskiego portalu ogłoszeniowego z nieruchomościami otodom.pl w formacie .json oraz przetworzone w języku Python i wyeksportowane do formatu tabelarycznego.

**Liczba obserwacji i zmiennych:** Zbiór zawiera 1251 obserwacji i 13 zmiennych.

**Krótki opis zmiennych:**

Zmienne ciągłe:
1. cena - całkowita wartość nieruchomości (zł)
2. powierzchnia - metraż mieszkania (m2)
3. czynsz - opłata administracyjna
4. odległość - dystans od centrum Warszawy (km)

Zmienne dyskretne:
1. wiek - ilość lat od teraz do roku wybudowania budynku
2. pokoje - liczba pomieszczeń mieszkalnych
3. piętro - numer piętra, na którym znajduje się lokal
4. wysokość - całkowita liczba pięter w budynku

Zmienne kategoryczne:
1. miejscowość - lokalizacja nieruchomości
2. rodzaj zabudowy - typ budynku (np. mieszkanie w bloku, mieszkanie w kamienicy, apartament)
3. stan wykończenia - np. do zamieszkania, do remontu
4. rodzaj okien - np. drewniane, plastikowe
5. rodzaj ogrzewania - np. miejskie, gazowe, własne

**Problem do rozwiązania:** Problem regresji – zmienną docelową jest cena nieruchomości, ciągła wartość liczbowa.

**Cel biznesowy projektu:** Stworzenie modelu wyceniającego nieruchomości na rynku wtórnym na terenie województwa mazowieckiego, który może zostać wykorzystany przez agentów nieruchomości, prywatnych inwestorów, a także klientów indywidualnych. Głównym założeniem projektu jest minimalizacja strat i ochrona przed „zamrożeniem” oferty w przypadku sprzedających oraz zabezpieczenie przed przepłaceniem za nieruchomość w przypadku nabywców.


#### **4. Podstawowe informacje o zbiorze danych**

Wczytanie zbioru danych oraz zaimportowanie potrzebnych bibliotek:

In [997]:
import pandas as pd
import numpy as np

df = pd.read_csv("dataset_csv.csv")
df

,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum,miasto,zabudowa,wykonczenie,okna,ogrzewanie
0,295000,54.26,73.00,686.00,2.00,1,3.00,91.95,Radom,blok,do remontu,plastikowe,miejskie
1,500000,27.76,55.00,300.00,1.00,5,10.00,3.79,Warszawa,blok,do zamieszkania,plastikowe,miejskie
2,790000,50.50,22.00,950.00,2.00,3,4.00,12.06,Warszawa,blok,do zamieszkania,plastikowe,miejskie
3,470000,32.50,27.00,NaN,1.00,2,3.00,12.02,Warszawa,blok,do zamieszkania,drewniane,miejskie
4,660000,41.23,3.00,750.00,2.00,parter,NaN,14.06,Ożarów Mazowiecki,blok,do zamieszkania,plastikowe,miejskie
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1246,790000,47.00,26.00,1200.00,2.00,4,11.00,5.40,Warszawa,blok,do wykończenia,NaN,NaN
1247,630000,58.50,50.00,1027.00,3.00,5,10.00,6.49,Warszawa,blok,do remontu,NaN,miejskie
1248,599000,36.70,62.00,600.00,2.00,2,4.00,2.42,Warszawa,blok,do zamieszkania,plastikowe,miejskie
1249,819000,53.70,65.00,873.00,3.00,7,7.00,3.63,Warszawa,blok,do remontu,NaN,miejskie


Liczebność zbioru danych oraz typy zmiennych:

In [998]:
size = df.shape[0]
print(f"Zbiór liczy {size} rekordów.")

Zbiór liczy 1251 rekordów.


In [999]:
types = pd.DataFrame({"Nazwa":df.columns, "Typ":df.dtypes, "Indeks":range(1, len(df.columns)+1)})
types_indexed = types.set_index("Indeks")
types_indexed

,Nazwa,Typ
Indeks,,
1,cena,int64
2,powierzchnia,float64
3,wiek,float64
4,czynsz,float64
5,pokoje,float64
6,piętro,object
7,wysokość budynku,float64
8,odległość od centrum,float64
9,miasto,object


Podstawowe statystyki dla zmiennych ciągłych:

In [1000]:
pd.options.display.float_format = '{:.2f}'.format
statistics = df.describe()
statistics

,cena,powierzchnia,wiek,czynsz,pokoje,wysokość budynku,odległość od centrum
count,1251.00,1251.00,1167.00,1038.00,1250.00,1210.00,1251.00
mean,1105884.19,60.41,35.19,1075.42,2.59,6.17,12.31
std,900983.04,30.98,64.69,4749.78,0.95,4.19,20.38
min,145000.00,19.00,-78.00,1.00,1.00,1.00,0.19
25%,659000.00,42.59,10.00,650.00,2.00,4.00,3.92
50%,850000.00,53.73,24.00,820.00,2.00,5.00,6.34
75%,1218500.00,68.87,56.00,1088.00,3.00,8.00,10.09
max,12900000.00,440.28,2021.00,153179.00,7.00,52.00,108.04


#### **5. Walidacja danych**

##### **5.1 Ujemne wartości w kolumnie wiek**

Przede wszystkim możemy zaobserwować, że najmniejsza wartość w kolumnie *wiek* jest ujemna. Z tego powodu musimy przyjrzeć się tej kolumnie szerzej: sprawdzic czy ma więcej ujemnych wartości oraz je obsłużyć.

In [1001]:
df[df["wiek"] < 0]

,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum,miasto,zabudowa,wykonczenie,okna,ogrzewanie
261,651280,46.52,-1.00,NaN,2.00,parter,2.00,13.01,Warszawa,blok,do wykończenia,plastikowe,miejskie
267,897328,65.98,-1.00,NaN,3.00,1,2.00,13.02,Warszawa,blok,do wykończenia,plastikowe,miejskie
715,820000,42.50,-78.00,695.00,2.00,parter,5.00,8.27,Warszawa,blok,do zamieszkania,plastikowe,miejskie


In [1002]:
df = df.drop(index=715)

Usnęłam komórkę z nielogicznie dużą wartością. Wartości -1 pozostawiam, możemy domyślać się po stanie wykończenia *do wykończenia*, że są to mieszkania zaplanowane do budowy na przyszły rok.

##### **5.2 Zmienna piętro jako typ numeryczny**

Zmienna *piętro*, ze względu na obecność wartości opisowych została pierwotnie wczytana jako typ tekstowy. Zamienię *parter* na 0, *suterenę* na -1, natomiast wartości *poddasze* oraz *>10* na wysokość konkretnego budynku. Następnie przeprowadzę konwersję tej kolumny na typ numeryczny, aby włączyć ją w dalsze obliczenia statystyczne.

In [1003]:
floor_mapping = {
    "parter":0,
    "suterena":-1}

df["piętro"] = df["piętro"].replace(floor_mapping)
df.loc[df["piętro"] == "> 10", "piętro"] = df["wysokość budynku"]
df.loc[df["piętro"] == "poddasze", "piętro"] = df["wysokość budynku"]
df["piętro"] = df["piętro"].astype(float)

In [1004]:
df["piętro"].dtype

dtype('float64')

##### **5.3 Statystyki**

Jeszcze raz wyświetlam statystyki z poprawionymi wartościami *wieku* oraz dodaną kolumną *piętro*.

In [1005]:
statistics = df.describe()
statistics

,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum
count,1250.00,1250.00,1166.00,1037.00,1249.00,1240.00,1209.00,1250.00
mean,1106112.90,60.43,35.29,1075.79,2.59,3.25,6.17,12.32
std,901307.32,30.99,64.64,4752.06,0.95,3.73,4.19,20.39
min,145000.00,19.00,-1.00,1.00,1.00,-1.00,1.00,0.19
25%,659000.00,42.69,10.00,650.00,2.00,1.00,4.00,3.92
50%,850000.00,53.75,24.00,820.00,2.00,2.00,5.00,6.33
75%,1218750.00,68.94,56.00,1092.00,3.00,4.00,8.00,10.09
max,12900000.00,440.28,2021.00,153179.00,7.00,52.00,52.00,108.04


#### **6. Analiza wartości odstających**

W celu identyfikacji wartości odstających, użyję metody rozstępu ćwiartkowego (IQR). Zamiast jednak *ucinać* dane na sztywno, przeanalizuję najpierw te skrajne przypadki i usunę wiersze, które nie mają logicznego sensu z punktu widzenia rynku nieruchomości (np. skrajnie niska cena za mieszkanie).

In [1006]:
def iqr(col):
    q1 = col.quantile(0.25)
    q3 = col.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr

    return lower, upper

##### **6.1 Cena**

In [1007]:
price_iqr = iqr(df["cena"])
print(price_iqr[:2])

print("Liczba wartości odstających:", df.loc[(df["cena"]<=price_iqr[0]) | (df["cena"]>=price_iqr[1]), "cena"].count())
df[(df["cena"]<=price_iqr[0]) | (df["cena"]>=price_iqr[1])].sort_values("cena")

(np.float64(-180625.0), np.float64(2058375.0))
Liczba wartości odstających: 108


,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum,miasto,zabudowa,wykonczenie,okna,ogrzewanie
879,2100000,77.18,8.00,NaN,3.00,1.00,4.00,4.52,Warszawa,apartamentowiec,do zamieszkania,drewniane,NaN
1162,2120000,125.25,26.00,1500.00,5.00,1.00,4.00,6.34,Warszawa,apartamentowiec,do zamieszkania,plastikowe,kotłownia
720,2150000,91.30,16.00,1200.00,4.00,2.00,10.00,9.90,Warszawa,blok,do zamieszkania,drewniane,miejskie
220,2150000,125.00,90.00,1500.00,5.00,2.00,2.00,3.35,Warszawa,kamienica,do remontu,plastikowe,miejskie
1026,2150000,91.00,26.00,1300.00,3.00,1.00,6.00,5.19,Warszawa,apartamentowiec,do zamieszkania,plastikowe,miejskie
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,6150000,150.00,8.00,2900.00,5.00,14.00,14.00,1.05,Warszawa,apartamentowiec,do zamieszkania,drewniane,miejskie
995,7550000,207.00,3.00,2500.00,4.00,4.00,4.00,3.91,Warszawa,apartamentowiec,do wykończenia,drewniane,miejskie
341,7900000,342.55,15.00,5500.00,4.00,8.00,8.00,0.89,Warszawa,apartamentowiec,NaN,NaN,miejskie
725,11990000,271.00,5.00,4000.00,7.00,7.00,8.00,1.38,Warszawa,apartamentowiec,do zamieszkania,plastikowe,miejskie


Jak możemy zauważyć, skrajnie wysokie ceny obejmują mieszkania o powierzchni w setkach $m^2$, w nowych apartamentowcach, usytuowanych blisko centrum Warszawy. Z tego powodu pozostawię te rekordy w bazie. 

##### **6.2 Powierzchnia**

In [1008]:
area_iqr = iqr(df["powierzchnia"])
print(area_iqr[:2])

print("Liczba wartości odstających: ", df.loc[(df["powierzchnia"]<=area_iqr[0]) | (df["powierzchnia"]>=area_iqr[1]), "powierzchnia"].count())
df[(df["powierzchnia"]<=area_iqr[0]) | (df["powierzchnia"]>=area_iqr[1])].sort_values("powierzchnia")

(np.float64(3.3100000000000023), np.float64(108.31))
Liczba wartości odstających:  80


,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum,miasto,zabudowa,wykonczenie,okna,ogrzewanie
638,3258000,108.60,22.00,2600.00,3.00,5.00,7.00,3.54,Warszawa,apartamentowiec,do zamieszkania,NaN,miejskie
636,1810000,109.70,26.00,3200.00,3.00,3.00,11.00,7.89,Warszawa,apartamentowiec,do zamieszkania,plastikowe,miejskie
532,865000,111.00,1.00,NaN,5.00,1.00,2.00,14.90,Marki,blok,do wykończenia,plastikowe,inne
282,2900000,111.70,12.00,NaN,4.00,9.00,9.00,3.78,Warszawa,apartamentowiec,do zamieszkania,drewniane,miejskie
402,4825000,112.24,8.00,NaN,4.00,14.00,14.00,1.02,Warszawa,apartamentowiec,do zamieszkania,aluminiowe,miejskie
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1179,5647680,235.32,19.00,2500.00,5.00,3.00,3.00,9.20,Warszawa,apartamentowiec,do zamieszkania,drewniane,miejskie
681,3200000,245.00,17.00,1388.00,5.00,3.00,3.00,10.24,Warszawa,apartamentowiec,do zamieszkania,aluminiowe,gazowe
725,11990000,271.00,5.00,4000.00,7.00,7.00,8.00,1.38,Warszawa,apartamentowiec,do zamieszkania,plastikowe,miejskie
341,7900000,342.55,15.00,5500.00,4.00,8.00,8.00,0.89,Warszawa,apartamentowiec,NaN,NaN,miejskie


In [1009]:
print(round(3522240/440.28, 2))

8000.0


Analiza rekordu o największej powierzchni budzi wątpliwości. Cena na poziomie 3,5 mln zł przekłada się na zaledwie 8000 zł/$1m^2$, co przy lokalizacji oddalonej o 2,5 km od centrum Warszawy jest wartością rażąco niską. Nawet uwzględniając wiek i rodzaj budynku (100-letnia kamienica) oraz stan *do wykończenia*, oferta ta drastycznie odbiega od pozostałych odstających rekordów - dużych, luksusowych mieszkań w nowych apartamentowcach, które osiągają wielokrotnie wyższe ceny jednostkowe. Ze względu na wysokie ryzyko, że rekord ten stanowi błąd w danych lub dotyczy specyficznej formy własności (np. sprzedaży udziałów), usunę go z bazy.

In [1010]:
df = df.drop(index=374)

##### **6.3 Wiek**

In [1011]:
age_iqr = iqr(df["wiek"])
print(age_iqr[:2])

print("Liczba wartości odstających: ", df.loc[(df["wiek"]<=age_iqr[0]) | (df["wiek"]>=age_iqr[1]), "wiek"].count())
df[(df["wiek"]<=age_iqr[0]) | (df["wiek"]>=age_iqr[1])].sort_values("wiek")

(np.float64(-59.0), np.float64(125.0))
Liczba wartości odstających:  6


,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum,miasto,zabudowa,wykonczenie,okna,ogrzewanie
1073,1899000,68.00,125.00,700.00,3.00,2.00,6.00,0.99,Warszawa,kamienica,do wykończenia,plastikowe,miejskie
85,799000,46.67,126.00,890.00,2.00,1.00,4.00,1.06,Warszawa,kamienica,NaN,NaN,miejskie
167,1479000,55.00,126.00,720.00,3.00,2.00,5.00,0.66,Warszawa,kamienica,do zamieszkania,plastikowe,miejskie
979,2499000,99.30,126.00,1980.00,4.00,3.00,4.00,1.32,Warszawa,kamienica,do zamieszkania,drewniane,miejskie
598,350000,60.00,136.00,300.00,2.00,2.00,NaN,95.69,Płock,kamienica,do zamieszkania,plastikowe,inne
226,350000,30.00,2021.00,600.00,1.00,1.00,4.00,15.06,Pruszków,blok,do remontu,plastikowe,miejskie


Rekord o najwyższym wieku budynku ma nierealistycznej wartość 2021 lat. Wynika to najprawdopodobniej z błędu podczas wprowadzania danych – wiek został omyłkowo wpisany w rok budowy przez autora ogłoszenia. Ze względu na brak możliwości jednoznacznego ustalenia faktycznego wieku nieruchomości, usunę ten wiersz. Pozostałe wartości odstające (budynki mające 125–136 lat) są wiarygodne jak na realia rynkowe zabytkowych kamienic.

In [1012]:
df = df.drop(index=226)

##### **6.4 Czynsz**

In [1013]:
rent_iqr = iqr(df["czynsz"])
print(rent_iqr[:2])

print("Liczba wartości odstających: ", df.loc[(df["czynsz"]<=rent_iqr[0]) | (df["czynsz"]>=rent_iqr[1]), "czynsz"].count())
df[(df["czynsz"]<=rent_iqr[0]) | (df["czynsz"]>=rent_iqr[1])].sort_values("czynsz")

(np.float64(-19.0), np.float64(1765.0))
Liczba wartości odstających:  53


,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum,miasto,zabudowa,wykonczenie,okna,ogrzewanie
557,1999000,104.08,5.00,1776.00,5.00,4.00,6.00,7.56,Warszawa,blok,do zamieszkania,NaN,miejskie
459,2570000,96.80,11.00,1793.00,3.00,4.00,4.00,8.55,Warszawa,blok,do zamieszkania,drewniane,miejskie
1180,3650000,122.00,6.00,1800.00,5.00,4.00,4.00,8.73,Warszawa,apartamentowiec,do zamieszkania,plastikowe,miejskie
697,2600000,89.76,10.00,1800.00,4.00,1.00,5.00,1.97,Warszawa,apartamentowiec,do zamieszkania,drewniane,miejskie
904,1850000,97.50,10.00,1800.00,4.00,1.00,5.00,6.76,Warszawa,blok,do zamieszkania,plastikowe,miejskie
481,2990000,81.00,1.00,1800.00,3.00,6.00,28.00,1.61,Warszawa,apartamentowiec,do zamieszkania,aluminiowe,miejskie
978,2800000,122.00,99.00,1800.00,4.00,4.00,4.00,3.16,Warszawa,kamienica,do zamieszkania,drewniane,inne
642,2190000,116.00,19.00,1800.00,4.00,3.00,7.00,4.59,Warszawa,blok,do zamieszkania,plastikowe,miejskie
667,2350000,116.42,2.00,1820.00,3.00,0.00,4.00,11.98,Warszawa,apartamentowiec,do zamieszkania,plastikowe,miejskie
1008,1352260,74.30,31.00,1820.00,3.00,2.00,6.00,2.52,Warszawa,apartamentowiec,do zamieszkania,plastikowe,miejskie


Na pewno pozbędę się rekordu o najwyższym czynszu - ma on nierealistyczną wartość 153tyś, która wynika najprawdopodobniej z błędu podczas wprowadzania danych do ogłoszenia. Pozostałe wartości są realistyczne: dotyczną lokali o powierzchni w setkach $m^2$ w nowoczesnych apartamentowcach, albo dotyczną dużych mieszkań w prestiżowych, przedwojennych kamienicach w centrum Warszawy.

In [1014]:
df = df.drop(index=813)

##### **6.5 Pokoje**

In [1015]:
rooms_iqr = iqr(df["pokoje"])
print(rooms_iqr[:2])

print("Liczba wartości odstających: ", df.loc[(df["pokoje"]<=rooms_iqr[0]) | (df["pokoje"]>=rooms_iqr[1]), "pokoje"].count())
df[(df["pokoje"]<=rooms_iqr[0]) | (df["pokoje"]>=rooms_iqr[1])].sort_values("pokoje")

(np.float64(0.5), np.float64(4.5))
Liczba wartości odstających:  40


,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum,miasto,zabudowa,wykonczenie,okna,ogrzewanie
5,1199900,125.20,5.00,NaN,5.00,1.00,2.00,15.03,Kobyłka,apartamentowiec,do zamieszkania,plastikowe,gazowe
585,3590000,135.69,NaN,NaN,5.00,1.00,2.00,5.78,Warszawa,blok,do zamieszkania,drewniane,NaN
619,1450000,67.00,9.00,NaN,5.00,1.00,3.00,2.72,Warszawa,apartamentowiec,do zamieszkania,plastikowe,miejskie
664,1450000,97.00,42.00,1580.00,5.00,3.00,3.00,4.51,Warszawa,blok,do remontu,NaN,miejskie
681,3200000,245.00,17.00,1388.00,5.00,3.00,3.00,10.24,Warszawa,apartamentowiec,do zamieszkania,aluminiowe,gazowe
708,2700000,124.05,115.00,NaN,5.00,5.00,6.00,0.84,Warszawa,kamienica,do zamieszkania,NaN,NaN
776,1928000,107.12,14.00,1550.00,5.00,1.00,5.00,7.93,Warszawa,blok,do zamieszkania,drewniane,miejskie
790,2050000,145.58,26.00,NaN,5.00,5.00,6.00,2.58,Warszawa,blok,NaN,plastikowe,miejskie
961,3551360,198.40,30.00,4565.00,5.00,3.00,6.00,7.35,Warszawa,blok,do zamieszkania,drewniane,miejskie
1000,4790000,186.00,114.00,1450.00,5.00,2.00,6.00,1.16,Warszawa,kamienica,do remontu,NaN,miejskie


W analizowanym zbiorze wartości odstające należą do przedziału od 5 do 7 pokoi. Wiekszość rekordów jednak jest wiarygodnych - to mieszkania o dużej powierzchni (100-300$m^2$). Uwagę zwracają jednak ogłoszenia oferujące 5 lub więcej pokoi na metrażu rzędu 50–70 $m^2$. Takie parametry mogą sugerować specyficzne nieruchomości przygotowane pod wynajem, gdzie przestrzeń została maksymalnie podzielona. Aby zweryfikować rynkową zasadność tych danych i ich wpływ na model, przeanalizuję średni metraż przypadającego na jeden pokój.

In [1016]:
rooms_outliers = df.loc[[619,527,412,499,916], ["powierzchnia", "pokoje"]]
rooms_outliers["średni metraż"] = rooms_outliers["powierzchnia"] / rooms_outliers["pokoje"]
rooms_outliers

,powierzchnia,pokoje,średni metraż
619,67.00,5.00,13.40
527,67.00,5.00,13.40
412,68.74,5.00,13.75
499,58.10,6.00,9.68
916,63.30,6.00,10.55


Istnieje ryzyko, że w tych rekordach liczba pokoi nie obejmuje kuchni i łazienki. Nawet jeśli to realne oferty pod wynajem, ich skrajnie niski metraż na pokój jest nietypowy dla rynku. Aby nie zaburzać korelacji w modelu i uniknąć błędnych danych, zdecyduje sie na usunięcie.

In [1017]:
df = df.drop(index=[619,527,412,499,916])

##### **6.6 Piętro**

In [1018]:
floor_iqr = iqr(df["piętro"])
print(floor_iqr[:2])

print("Liczba wartości odstających: ", df.loc[(df["piętro"]<=floor_iqr[0]) | (df["piętro"]>=floor_iqr[1]), "piętro"].count())
df[(df["piętro"]<=floor_iqr[0]) | (df["piętro"]>=floor_iqr[1])].sort_values("piętro")

(np.float64(-3.5), np.float64(8.5))
Liczba wartości odstających:  69


,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum,miasto,zabudowa,wykonczenie,okna,ogrzewanie
1237,1249000,88.00,34.00,1400.00,4.00,9.00,10.00,6.23,Warszawa,blok,do zamieszkania,plastikowe,miejskie
900,730000,47.51,52.00,895.00,2.00,9.00,11.00,2.04,Warszawa,blok,do remontu,plastikowe,miejskie
921,399000,30.00,54.00,650.00,1.00,9.00,10.00,6.35,Warszawa,blok,do remontu,plastikowe,miejskie
932,1390000,76.48,12.00,1112.00,4.00,9.00,10.00,7.08,Warszawa,apartamentowiec,do zamieszkania,drewniane,miejskie
934,2695000,71.00,1.00,1065.00,3.00,9.00,9.00,1.69,Warszawa,apartamentowiec,do zamieszkania,drewniane,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1035,1297000,35.90,1.00,NaN,1.00,28.00,28.00,1.58,Warszawa,apartamentowiec,do zamieszkania,aluminiowe,miejskie
1149,1591000,37.00,2.00,NaN,1.00,29.00,29.00,1.60,Warszawa,apartamentowiec,do zamieszkania,NaN,NaN
987,795000,36.20,21.00,1000.00,1.00,30.00,30.00,1.23,Warszawa,blok,do zamieszkania,plastikowe,miejskie
1199,2890000,53.00,12.00,2200.00,2.00,44.00,44.00,0.50,Warszawa,apartamentowiec,NaN,NaN,miejskie


Mimo że metoda IQR wskazuje wartości odstające już od 9. piętra, bloki o wysokości nawet 15–20 pięter są standardem na wielu miejskich osiedlach. Natomiast mieszkania usytuowane na 30., 40. lub nawet 50. piętrze to apartamenty w wieżowcach mieszkalnych. Dane te są rynkowo poprawne, dlatego zdecydowałam się o zachowaniu wszystkich rekordów.

##### **6.7 Liczba pięter budynku**

In [1019]:
total_floors_iqr = iqr(df["wysokość budynku"])
print(total_floors_iqr[:2])

print("Liczba wartości odstających: ", df.loc[(df["wysokość budynku"]<=total_floors_iqr[0]) | (df["wysokość budynku"]>=total_floors_iqr[1]), "wysokość budynku"].count())
df[(df["wysokość budynku"]<=total_floors_iqr[0]) | (df["wysokość budynku"]>=total_floors_iqr[1])].sort_values("wysokość budynku")

(np.float64(-2.0), np.float64(14.0))
Liczba wartości odstających:  58


,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum,miasto,zabudowa,wykonczenie,okna,ogrzewanie
448,770000,36.07,60.00,630.00,2.00,14.00,14.00,1.95,Warszawa,blok,do zamieszkania,plastikowe,inne
1195,6150000,150.00,8.00,2900.00,5.00,14.00,14.00,1.05,Warszawa,apartamentowiec,do zamieszkania,drewniane,miejskie
397,869000,43.50,14.00,900.00,2.00,2.00,14.00,3.49,Warszawa,apartamentowiec,do zamieszkania,drewniane,miejskie
902,950000,54.80,16.00,960.00,2.00,14.00,14.00,6.75,Warszawa,blok,do zamieszkania,drewniane,miejskie
313,1242000,46.00,17.00,NaN,2.00,3.00,14.00,0.59,Warszawa,apartamentowiec,do zamieszkania,drewniane,miejskie
402,4825000,112.24,8.00,NaN,4.00,14.00,14.00,1.02,Warszawa,apartamentowiec,do zamieszkania,aluminiowe,miejskie
491,1380000,71.54,17.00,1070.00,2.00,8.00,15.00,7.01,Warszawa,apartamentowiec,do zamieszkania,NaN,miejskie
595,599000,27.40,54.00,650.00,1.00,15.00,15.00,0.58,Warszawa,blok,do remontu,NaN,miejskie
621,840000,56.80,43.00,857.00,3.00,6.00,15.00,6.27,Warszawa,blok,do zamieszkania,plastikowe,miejskie
623,1288000,73.00,6.00,1500.00,3.00,7.00,15.00,9.99,Warszawa,apartamentowiec,do zamieszkania,plastikowe,NaN


Podobnie jak w podpunkcie wyżej, wartości odstające wyznaczone są już od 14. piętra. Z perspektywy rynku wiemy, że budynki o wysokości 15–20 pięter są standardem, natomiast obiekty w przedziale 20–52 pięter to autentyczne apartamentowce w centrum Warszawy. Tutaj również zdecydowałam się o zachowaniu wszystkich rekordów.

##### **6.8 Odległość od centrum Warszawy**

In [1020]:
distance_iqr = iqr(df["odległość od centrum"])
print(distance_iqr[:2])

print("Liczba wartości odstających: ", df.loc[(df["odległość od centrum"]<=distance_iqr[0]) | (df["odległość od centrum"]>=distance_iqr[1]), "odległość od centrum"].count())
df[(df["odległość od centrum"]<=distance_iqr[0]) | (df["odległość od centrum"]>=distance_iqr[1])].sort_values("odległość od centrum")

(np.float64(-5.334999999999999), np.float64(19.345))
Liczba wartości odstających:  132


,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum,miasto,zabudowa,wykonczenie,okna,ogrzewanie
808,450000,46.25,NaN,600.00,3.00,3.00,4.00,19.72,Legionowo,blok,do remontu,plastikowe,miejskie
251,175000,21.00,87.00,170.00,1.00,2.00,2.00,19.98,Wołomin,kamienica,do remontu,plastikowe,elektryczne
483,660000,47.00,20.00,NaN,3.00,1.00,3.00,20.29,Wołomin,blok,do zamieszkania,plastikowe,gazowe
298,650000,55.00,0.00,NaN,2.00,0.00,1.00,20.74,Legionowo,apartamentowiec,do wykończenia,plastikowe,gazowe
300,800000,59.49,6.00,950.00,3.00,4.00,4.00,21.06,Brwinów,blok,do zamieszkania,NaN,inne
...,...,...,...,...,...,...,...,...,...,...,...,...,...
924,409000,72.90,26.00,773.00,3.00,2.00,3.00,105.46,Mława,blok,do zamieszkania,plastikowe,kotłownia
601,459000,60.45,15.00,318.00,4.00,1.00,3.00,106.97,Mława,blok,do zamieszkania,plastikowe,gazowe
494,495000,50.20,2.00,450.00,2.00,4.00,4.00,107.38,Mława,blok,do zamieszkania,plastikowe,gazowe
237,499000,60.64,9.00,NaN,3.00,1.00,4.00,107.73,Mława,blok,do zamieszkania,plastikowe,gazowe


Wartości odstające dla *odległości od centrum* dotyczą lokali położonych poza granicami Warszawy. Ich autentyczność potwierdzają adekwatnie niższe ceny rynkowe. Ponieważ rekordy te są poprawne i odzwierciedlają realne zróżnicowanie lokalizacji w zbiorze, jak najbardziej pozostawiam je w bazie.

#### **6. Analiza brakujących wartości**

Identyfikacja kolumn z brakującymi wartościami oraz ich udział procentowy:

In [1021]:
missing_vals = df.isnull().sum()
missing_vals_df = pd.DataFrame({"braki":missing_val})
missing_vals_df["% udział"] = missing_vals_df["braki"]/rozmiar * 100
missing_vals_df

,braki,% udział
cena,0,0.00
powierzchnia,0,0.00
wiek,84,6.71
czynsz,212,16.95
pokoje,0,0.00
piętro,10,0.80
wysokość budynku,41,3.28
odległość od centrum,0,0.00
miasto,0,0.00
zabudowa,0,0.00


Dla przypomnienia oraz dalszej analizy wyświetlimy jeszcze raz podstawowe statystyki, już bez wartości odstających.

In [1022]:
statistics = df.describe()
statistics

,cena,powierzchnia,wiek,czynsz,pokoje,piętro,wysokość budynku,odległość od centrum
count,1242.00,1242.00,1158.00,1030.00,1242.00,1232.00,1201.00,1242.00
mean,1104647.25,60.11,33.50,928.70,2.58,3.25,6.18,12.36
std,900987.82,29.13,28.12,479.44,0.93,3.74,4.20,20.45
min,145000.00,19.00,-1.00,1.00,1.00,-1.00,1.00,0.19
25%,659000.00,42.54,10.00,650.00,2.00,1.00,4.00,3.92
50%,850000.00,53.65,24.00,820.00,2.00,2.00,5.00,6.33
75%,1216000.00,68.91,56.00,1074.50,3.00,4.00,8.00,10.09
max,12900000.00,342.55,136.00,5500.00,7.00,52.00,52.00,108.04


##### **6.1 Wiek**

Braki w kolumnie *wiek* stanowią niecałe 7% wartości zmiennej. Uzupełnię je na podstawie średniego wieku dla każdego rodzaju zabudowy.

In [1023]:
df["wiek"].isnull().sum()

np.int64(84)

In [1024]:
df.loc[df["wiek"].isnull(), "zabudowa"].value_counts()

zabudowa
blok               60
apartamentowiec    12
kamienica          12
Name: count, dtype: int64

In [1025]:
df.groupby("zabudowa")["wiek"].mean().round(0)

zabudowa
apartamentowiec   11.00
blok              33.00
kamienica         84.00
Name: wiek, dtype: float64

In [1026]:
df["wiek"] = df["wiek"].fillna(df.groupby("zabudowa")["wiek"].transform("mean").round(0))

In [1027]:
df["wiek"].isnull().sum()

np.int64(0)

##### **6.2 Czynsz**

Braki w kolumnie *czynsz* stanowią prawie 17% wartości zmiennej. Aby uzupełnić te dane w sposób logiczny i niezaburzający dalszej analizy, wykorzystam dostępne dane o powierzchni nieruchomości, zachowując proporcję pomiędzy wielkością mieszkania, a jego kosztami administracyjnymi. Wyznaczam współczynnik *factor*, reprezentujący średni czynsz na $1 m^2$.

In [1028]:
factor = (df["czynsz"]/df["powierzchnia"]).mean()
factor

np.float64(16.003688941936577)

*Wsp* wynosi ok. 16 zł za $1 m^2$. 
W kolejnym kroku uzupełnię brakujące wartości w kolumnie *czynsz*, mnożąc ten współczynnik przez powierzchnię danej nieruchomości.

In [1029]:
df.loc[df["czynsz"].isnull(), "czynsz"] = df.loc[df["czynsz"].isnull(), "powierzchnia"] * factor

In [1030]:
df["czynsz"].isnull().sum()

np.int64(0)

##### **6.3 Piętro**

Braki w kolumnie *piętro* stanowią niespełna 1% wartości zmiennej. Ze względu na znikomy udział, zdecydowałam o ich uzupełnięniu medianą, co pozwoli zachować kompletność zbioru bez ryzyka zniekształcenia danych.

In [1031]:
floor_median = df["piętro"].median()
floor_median

2.0

In [1032]:
df["piętro"] = df["piętro"].fillna(floor_median)

In [1033]:
df["piętro"].isnull().sum()

np.int64(0)

##### **6.4 Wysokość budynku**

Braki w kolumnie *wysokość budynku* stanowią lekko ponad 3% wartości zmiennej. Uzupełnie je średnią wartością tej kolumny dla rodzaju zabudowy. Dodatkowo wprowadzę jednak warunek korygujący: w przypadkach, gdy przypisana mediana będzie niższa niż wartość w kolumnie *piętro*, wysokość budynku zrównam z kondygnacją lokalu.

In [1034]:
df.loc[df["wysokość budynku"].isnull(), "zabudowa"].value_counts()

zabudowa
blok               29
apartamentowiec     7
kamienica           5
Name: count, dtype: int64

In [1035]:
df.groupby("zabudowa")["wysokość budynku"].mean().round(0)

zabudowa
apartamentowiec   7.00
blok              6.00
kamienica         4.00
Name: wysokość budynku, dtype: float64

In [1036]:
df["wysokość budynku"] = df["wysokość budynku"].fillna(df.groupby("zabudowa")["wysokość budynku"].transform("mean").round(0))

In [1037]:
df["wysokość budynku"].isnull().sum()

np.int64(0)

##### **6.5 Wykończenie**

Braki w kolumnie wykończenie stanowią ok. 8% wartości zmiennej. Uzupełniłam je najczęstszą wartością z całego zbioru, czyli modą. Przy tak małym odsetku braków to najszybsze i w pełni bezpieczne rozwiązanie, które pozwala domknąć bazę bez sztucznego komplikowania sprawy. Ciężko bowiem znaleźć tu silne korelacje z innymi zmiennymi, a samych stanów wykończenia nie da się łatwo pogrupować czy uszeregować (nie zawsze obiektywnie wiadomo, co jest lepsze, a co gorsze).

In [1038]:
df["wykonczenie"].value_counts()

wykonczenie
do zamieszkania    907
do remontu         144
do wykończenia      89
Name: count, dtype: int64

In [1039]:
condition_mode = df["wykonczenie"].mode()[0]
condition_mode

'do zamieszkania'

In [1040]:
df["wykonczenie"] = df["wykonczenie"].fillna(condition_mode)

In [1041]:
df["wykonczenie"].isnull().sum()

np.int64(0)

##### **6.6 Okna**

In [1042]:
windows_and_age_sorted = df[["wiek", "okna"]].sort_values("wiek")

In [1043]:
windows_and_age_sorted.head(200)["okna"].value_counts()

okna
plastikowe    123
drewniane      20
aluminiowe      2
Name: count, dtype: int64

In [1044]:
windows_and_age_sorted.tail(200)["okna"].value_counts()

okna
plastikowe    115
drewniane      27
Name: count, dtype: int64

Ze względu na wysoki odsetek braków danych (28%) oraz brak wiarygodnej metody ich estymacji, kolumna *okna* zostanie całkowicie usunięta ze zbioru. Wstępna analiza nie wykazuje wyraźnej zależności między rodzajem stolarki okiennej a wiekiem budynku. W starszych budynkach drewniane okna są masowo wymieniane na plastikowe (PCV), podczas gdy w nowych inwestycjach klasy premium często powraca się do okien drewnianych. Chociąz widać, że na rynku dominuje plastik, odgórne uzupełnienie tą wartością blisko jednej trzeciej bazy sztucznie zniekształciłoby dane.

In [1045]:
df = df.drop("okna", axis=1)

##### **6.7 Ogrzewanie**

In [1046]:
df["ogrzewanie"].value_counts()

ogrzewanie
miejskie       994
gazowe          86
kotłownia       42
inne            28
elektryczne      7
Name: count, dtype: int64

In [1047]:
df.groupby("zabudowa")["ogrzewanie"].describe()["top"]

zabudowa
apartamentowiec    miejskie
blok               miejskie
kamienica          miejskie
Name: top, dtype: object

In [1048]:
df.groupby(pd.qcut(df['wiek'], q=4))["ogrzewanie"].describe()["top"]

C:\Users\oliwi\AppData\Local\Temp\ipykernel_28540\3109487941.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.qcut(df['wiek'], q=4))["ogrzewanie"].describe()["top"]


wiek
(-1.001, 11.0]    miejskie
(11.0, 26.0]      miejskie
(26.0, 55.0]      miejskie
(55.0, 136.0]     miejskie
Name: top, dtype: object

Podobnie jak w przypadku zmiennej *okna*, estymacja rodzaju ogrzewania na podstawie innych zmiennych okazała się trudna, ponieważ na analizowanym rynku dominuje ogrzewanie miejskie. Jednak w przeciwieństwie do stolarki okiennej, braki danych w kolumnie ogrzewanie stanowią zaledwie 6%, dlatego uzupełnie je najczęściej występującą wartością (modą).

In [1049]:
heating_mode = df["ogrzewanie"].mode()[0]
heating_mode

'miejskie'

In [1050]:
df["ogrzewanie"] = df["ogrzewanie"].fillna(heating_mode)

In [1051]:
df["ogrzewanie"].isnull().sum()

np.int64(0)